In [1]:
# Set the base dir to the project root
%cd ..
!pwd

/mnt/remote/workspaces/arthur.babey/project/FoundedPBI-code
/mnt/remote/workspaces/arthur.babey/project/FoundedPBI-code


In [2]:
import torch
import pandas as pd
import os
from pbi_utils.data_manager import H5pyEmbeddingsManager, PerphectDataInput
from main import parse_config, reduce_dimensionality, test_model, make_dataset
import numpy as np
import matplotlib.pyplot as plt

In [3]:
config = parse_config(config_path="output/pbip/best/training_config.yaml")

device = "cpu"

[DEBUG] [DNABERT2] Max sequence length for DNABERT2: 16384
[DEBUG] [DNABERT2] Max sequence length for DNABERT2: 16384
[INFO] Configuration loaded from output/pbip/best/training_config.yaml: Config(input_perphect=bacteria_df='data/perphect-data/predphi/bacteria_sequences.csv' phages_df='data/perphect-data/predphi/bacteriophages_sequences.csv' couples_df='data/perphect-data/predphi/predphi_train_dataset.csv', embeddings_dir=data/embeddings-pbip, num_gpu=0, gpu_id=0, training_config=TrainingConfig(do_train=True do_test=True epochs=100 batch_size=256 learning_rate=0.001 weight_decay=0.0001 training_noise_std=0.05 stratify_cv=True k_folds_cv=10 patience_early_stopping=1000 monitor_metric_early_stopping='f1' patience_reduce_lr=1000 monitor_metric_reduce_lr='f1' multiplying_factor_reduce_lr=0.5 reduce_dimensionality='none' n_components_bacteria=None n_components_phages=None), phages_embedding_models=[NT2(merging_strategy=TopBottomTruncateStrategy(), overlap=0, model_name='nucleotide-transform

In [4]:
# Hardcode test dataset path. Only used for the PredPHI data, on the CI4CB data, ignore this cell.
config.input_perphect.couples_df = "data/perphect-data/predphi/predphi_test_dataset.csv"

In [5]:
# Assumming that all the embeddings have already been computed
output_manager = H5pyEmbeddingsManager(config.embeddings_dir)

[INFO] Embeddings will be stored or read from data/embeddings-pbip


In [6]:
bacteria_df, phages_df, couples_df = PerphectDataInput(input_paths=config.input_perphect).load()

[INFO] Perphect input files will be read from data/perphect-data/predphi/bacteria_sequences.csv, data/perphect-data/predphi/bacteriophages_sequences.csv and data/perphect-data/predphi/predphi_test_dataset.csv
[INFO] Reading csv files...


In [7]:
bacteria_model_names = [x.name() for x in config.bacteria_embedding_models]
phages_model_names = [x.name() for x in config.phages_embedding_models]
dataset = make_dataset(couples_df, bacteria_model_names, phages_model_names, output_manager, device)
dataset = reduce_dimensionality(dataset, config.training_config.reduce_dimensionality, config.output_dir, config.training_config.n_components_bacteria, config.training_config.n_components_phages)


[INFO] Creating dataset (loading embeddings)...
[DEBUG] Loading 1236 embeddings for model NT2-TruncateStrategy-250M-ov0 from data/embeddings-pbip


Loading embeddings:   0%|          | 0/1236 [00:00<?, ?it/s]

[DEBUG] Loading 1236 embeddings for model MegaDNA-TruncateStrategy-concat-ov0 from data/embeddings-pbip


Loading embeddings:   0%|          | 0/1236 [00:00<?, ?it/s]

[DEBUG] Loading 1236 embeddings for model DNABERT2-BottomTruncateStrategy-ov0-maxlen16384 from data/embeddings-pbip


...

[DEBUG] Loading 1236 embeddings for model MegaDNA-TopBottomTruncateStrategy-concat-ov0 from data/embeddings-pbip


Loading embeddings:   0%|          | 0/1236 [00:00<?, ?it/s]

[DEBUG] Loading 1236 embeddings for model DNABERT2-TKPert-concat-J16-g20-ov0-maxlen16384 from data/embeddings-pbip


Loading embeddings:   0%|          | 0/1236 [00:00<?, ?it/s]

[DEBUG] Final embedding size (bacteria): 2500
[DEBUG] Final embedding size (phages): 15752


In [8]:
model_path = os.path.join(config.output_dir, "trained_model.pth")

In [9]:
bacterium_embed_size = len(dataset["bacterium_embedding"].iloc[0])
phage_embed_size = len(dataset["phage_embedding"].iloc[0])
model = config.classifier(bacterium_embed_size, phage_embed_size, **config.classifier_params)

model.load_state_dict(torch.load(model_path, map_location=device))

<All keys matched successfully>

In [10]:
test_model(dataset, model, batch_size=config.training_config.batch_size, device=device);

[INFO] Starting testing...
[INFO] Accuracy (test): 0.7508090734481812
[INFO] Recall (test): 0.7750809192657471
[INFO] F1 score (test): 0.7567140460014343
[INFO] Loss (test): 0.7344119245952001
[INFO] Confusion Matrix (test) (TP, FP, FN, TN): (479, 169, 139, 449)


In [11]:
dataset[["prediction", "probability"]] = dataset.apply(
    lambda row: pd.Series({
        "prediction": int(torch.argmax(torch.sigmoid(model(
            row["bacterium_embedding"].unsqueeze(0).to(device),
            row["phage_embedding"].unsqueeze(0).to(device)
        ))).squeeze().detach().cpu().numpy()),
        "probability": float(torch.max(torch.sigmoid(model(
            row["bacterium_embedding"].unsqueeze(0).to(device),
            row["phage_embedding"].unsqueeze(0).to(device)
        ))).squeeze().detach().cpu().numpy())
    }), axis=1
)
dataset.head()

,phage_id,bacterium_id,interaction_type,bacterium_embedding,phage_embedding,prediction,probability
0,NC_019400,CALF01000071,1,"[tensor(0.0652), tensor(0.0152), tensor(0.2972...","[tensor(0.1758), tensor(0.2331), tensor(0.2490...",0.0,0.750215
1,MF919516,CP012196,0,"[tensor(-0.0643), tensor(-0.0283), tensor(0.82...","[tensor(0.3073), tensor(0.4169), tensor(0.5371...",0.0,0.770389
2,NC_019813,CP011132,0,"[tensor(0.0619), tensor(0.1624), tensor(0.7857...","[tensor(0.0379), tensor(0.1556), tensor(0.2032...",0.0,0.699604
3,NC_027355,CM000736,1,"[tensor(-0.0317), tensor(-0.2202), tensor(0.08...","[tensor(0.0881), tensor(-0.1265), tensor(0.036...",0.0,0.805603
4,NC_021790,CP009976,1,"[tensor(-0.1541), tensor(-0.6002), tensor(0.02...","[tensor(0.0610), tensor(0.0364), tensor(0.3254...",1.0,0.908941


In [12]:
from Bio import Entrez
import xml.etree.ElementTree as ET
import time

Entrez.email = "arthur.babey@heig-vd.ch"  # Required by NCBI

RANKS = ["superkingdom", "phylum", "class", "order", "family", "genus"]

def get_taxids(accession_ids, batch_size=20):
    """Map accession IDs -> TaxIDs via nuccore docsum XML."""
    taxid_map = {}
    for i in range(0, len(accession_ids), batch_size):
        batch = accession_ids[i:i + batch_size]
        handle = Entrez.efetch(db="nuccore", id=",".join(batch), rettype="docsum", retmode="xml")
        root = ET.parse(handle).getroot()
        handle.close()
        for doc in root.iter("DocSum"):
            acc = doc.find(".//Item[@Name='Caption']")
            taxid = doc.find(".//Item[@Name='TaxId']")
            if acc is not None and taxid is not None and taxid.text:
                taxid_map[acc.text] = taxid.text
        time.sleep(0.34)
    return taxid_map

def get_taxonomy(taxid_map, batch_size=20):
    """Map TaxIDs -> dict of taxonomic ranks via taxonomy DB."""
    unique_taxids = [t for t in set(taxid_map.values()) if t is not None]
    tax_map = {}
    for i in range(0, len(unique_taxids), batch_size):
        batch = unique_taxids[i:i + batch_size]
        handle = Entrez.efetch(db="taxonomy", id=",".join(batch), retmode="xml")
        root = ET.parse(handle).getroot()
        handle.close()
        for taxon in root.iter("Taxon"):
            taxid = taxon.findtext("TaxId")
            lineage = {"species": taxon.findtext("ScientificName")}
            for lineage_node in taxon.iter("LineageEx"):
                for t in lineage_node.iter("Taxon"):
                    rank = t.findtext("Rank")
                    name = t.findtext("ScientificName")
                    if rank in RANKS:
                        lineage[rank] = name
            tax_map[taxid] = lineage
        time.sleep(0.34)
    return tax_map

# --- Per-bacterium counts split by correctness and interaction type ---
errors  = dataset[dataset["interaction_type"] != dataset["prediction"]]
correct = dataset[dataset["interaction_type"] == dataset["prediction"]]

def counts_by_type(df, label):
    return df[df["interaction_type"] == label]["bacterium_id"].value_counts()

total_pos   = counts_by_type(dataset, 1)
total_neg   = counts_by_type(dataset, 0)
errors_pos  = counts_by_type(errors,  1)   # FN
errors_neg  = counts_by_type(errors,  0)   # FP
correct_pos = counts_by_type(correct, 1)   # TP
correct_neg = counts_by_type(correct, 0)   # TN

total_counts  = dataset["bacterium_id"].value_counts()
error_counts  = errors["bacterium_id"].value_counts()
accessions    = total_counts.index.tolist()  # all bacteria, including those with 0 errors

print(f"Fetching taxonomy for {len(accessions)} unique bacteria accessions...")
taxid_map = get_taxids(accessions)
missing = [acc for acc in accessions if acc not in taxid_map]
if missing:
    print(f"Warning: no TaxID found for {len(missing)} accessions: {missing}")
tax_map = get_taxonomy(taxid_map)
print("Done.")

rows = []
for acc in accessions:
    taxid = taxid_map.get(acc)
    tax   = tax_map.get(taxid, {}) if taxid else {}

    tot      = total_counts.get(acc, 0)
    tot_pos  = total_pos.get(acc, 0)
    tot_neg  = total_neg.get(acc, 0)
    err      = error_counts.get(acc, 0)
    err_pos  = errors_pos.get(acc, 0)
    err_neg  = errors_neg.get(acc, 0)
    cor_pos  = correct_pos.get(acc, 0)
    cor_neg  = correct_neg.get(acc, 0)
    cor      = cor_pos + cor_neg

    rows.append({
        "bacterium_id": acc,
        # overall
        "errors":       err,
        "correct":      cor,
        "total":        tot,
        "error_rate":   round(err / tot, 3) if tot else None,
        "correct_rate": round(cor / tot, 3) if tot else None,
        # positive interactions
        "errors_pos":   err_pos,   # FN
        "correct_pos":  cor_pos,   # TP
        "total_pos":    tot_pos,
        "fn_rate":      round(err_pos / tot_pos, 3) if tot_pos else None,
        "tp_rate":      round(cor_pos / tot_pos, 3) if tot_pos else None,
        # negative interactions
        "errors_neg":   err_neg,   # FP
        "correct_neg":  cor_neg,   # TN
        "total_neg":    tot_neg,
        "fp_rate":      round(err_neg / tot_neg, 3) if tot_neg else None,
        "tn_rate":      round(cor_neg / tot_neg, 3) if tot_neg else None,
        # taxonomy
        "species":      tax.get("species",      "N/A"),
        "genus":        tax.get("genus",        "N/A"),
        "family":       tax.get("family",       "N/A"),
        "order":        tax.get("order",        "N/A"),
        "class":        tax.get("class",        "N/A"),
        "phylum":       tax.get("phylum",       "N/A"),
        "superkingdom": tax.get("superkingdom", "N/A"),
    })

taxonomy_df = pd.DataFrame(rows)
taxonomy_df


Fetching taxonomy for 107 unique bacteria accessions...
Done.


,bacterium_id,errors,correct,total,error_rate,correct_rate,errors_pos,correct_pos,total_pos,fn_rate,...,total_neg,fp_rate,tn_rate,species,genus,family,order,class,phylum,superkingdom
0,HG530068,53,48,101,0.525,0.475,49,48,97,0.505,...,4,1.000,0.000,Pseudomonas aeruginosa PA38182,Pseudomonas,Pseudomonadaceae,Pseudomonadales,Gammaproteobacteria,Pseudomonadota,N/A
1,CP002815,1,82,83,0.012,0.988,0,80,80,0.000,...,3,0.333,0.667,Cutibacterium acnes 6609,Cutibacterium,Propionibacteriaceae,Propionibacteriales,Actinomycetes,Actinomycetota,N/A
2,AP009493,4,66,70,0.057,0.943,3,61,64,0.047,...,6,0.167,0.833,Streptomyces griseus subsp. griseus NBRC 13350,Streptomyces,Streptomycetaceae,Kitasatosporales,Actinomycetes,Actinomycetota,N/A
3,JTHH01000008,7,28,35,0.200,0.800,6,25,31,0.194,...,4,0.250,0.750,Bacillus thuringiensis serovar morrisoni,Bacillus,Bacillaceae,Bacillales,Bacilli,Bacillota,N/A
4,CP010812,12,16,28,0.429,0.571,12,14,26,0.462,...,2,0.000,1.000,Vibrio cholerae,Vibrio,Vibrionaceae,Vibrionales,Gammaproteobacteria,Pseudomonadota,N/A
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
102,CP014518,2,1,3,0.667,0.333,0,1,1,0.000,...,2,1.000,0.000,Sinomonas atrocyanea,Sinomonas,Micrococcaceae,Micrococcales,Actinomycetes,Actinomycetota,N/A
103,JN632628,0,3,3,0.000,1.000,0,1,1,0.000,...,2,0.000,1.000,Connochaetes taurinus,Connochaetes,Bovidae,Artiodactyla,Mammalia,Chordata,N/A
104,CALB01000097,2,1,3,0.667,0.333,0,1,1,0.000,...,2,1.000,0.000,Cronobacter turicensis 564,Cronobacter,Enterobacteriaceae,Enterobacterales,Gammaproteobacteria,Pseudomonadota,N/A
105,CP016889,0,3,3,0.000,1.000,0,1,1,0.000,...,2,0.000,1.000,Pantoea agglomerans,Pantoea,Erwiniaceae,Enterobacterales,Gammaproteobacteria,Pseudomonadota,N/A


In [13]:
taxonomy_df.to_csv('./error_analysis_by_bacterium.csv', index=False)

In [14]:
# --- Aggregate by taxonomic rank ---
def rank_summary(df, rank):
    grp = df.groupby(rank).agg(
        total_pairs=("total", "sum"),
        # errors
        total_errors=("errors", "sum"),
        total_errors_pos=("errors_pos", "sum"),
        total_errors_neg=("errors_neg", "sum"),
        total_pairs_pos=("total_pos", "sum"),
        total_pairs_neg=("total_neg", "sum"),
        # correct
        total_correct=("correct", "sum"),
        total_correct_pos=("correct_pos", "sum"),
        total_correct_neg=("correct_neg", "sum"),
        n_species=("species", "nunique"),
    ).reset_index()
    grp["error_rate"]   = (grp["total_errors"]       / grp["total_pairs"]).round(3)
    grp["correct_rate"] = (grp["total_correct"]      / grp["total_pairs"]).round(3)
    grp["fn_rate"]      = (grp["total_errors_pos"]   / grp["total_pairs_pos"].replace(0, float("nan"))).round(3)
    grp["tp_rate"]      = (grp["total_correct_pos"]  / grp["total_pairs_pos"].replace(0, float("nan"))).round(3)
    grp["fp_rate"]      = (grp["total_errors_neg"]   / grp["total_pairs_neg"].replace(0, float("nan"))).round(3)
    grp["tn_rate"]      = (grp["total_correct_neg"]  / grp["total_pairs_neg"].replace(0, float("nan"))).round(3)
    return grp.sort_values("total_errors", ascending=False)

DISPLAY_COLS = [
    "{rank}",
    "n_species", "total_pairs",
    "total_errors", "error_rate", "total_correct", "correct_rate",
    "fn_rate", "tp_rate",
    "fp_rate", "tn_rate",
]

for rank in ["phylum", "class", "order", "family"]:
    print(f"\n=== By {rank} ===")
    summary = rank_summary(taxonomy_df, rank)
    cols = [c.format(rank=rank) if "{rank}" in c else c for c in DISPLAY_COLS]
    display(summary[cols].head(40).sort_values("error_rate", ascending=False))



=== By phylum ===


,phylum,n_species,total_pairs,total_errors,error_rate,total_correct,correct_rate,fn_rate,tp_rate,fp_rate,tn_rate
6,Chlamydiota,2,9,5,0.556,4,0.444,0.000,1.000,0.714,0.286
15,Myxococcota,1,10,4,0.400,6,0.600,0.000,1.000,0.444,0.556
18,Rhodothermota,1,5,2,0.400,3,0.600,0.000,1.000,0.500,0.500
13,Methanobacteriota,3,23,9,0.391,14,0.609,0.167,0.833,0.471,0.529
10,Deinococcota,1,11,4,0.364,7,0.636,0.333,0.667,0.375,0.625
9,Cyanobacteriota,3,36,13,0.361,23,0.639,0.385,0.615,0.300,0.700
17,Pseudomonadota,40,461,153,0.332,308,0.668,0.386,0.614,0.276,0.724
5,Campylobacterota,1,10,3,0.300,7,0.700,0.500,0.500,0.250,0.750
1,Arthropoda,2,11,3,0.273,8,0.727,1.000,0.000,0.111,0.889
12,Haptophyta,2,13,3,0.231,10,0.769,0.000,1.000,0.273,0.727



=== By class ===


,class,n_species,total_pairs,total_errors,error_rate,total_correct,correct_rate,fn_rate,tp_rate,fp_rate,tn_rate
6,Chlamydiia,2,9,5,0.556,4,0.444,0.000,1.000,0.714,0.286
20,Methanobacteria,1,5,2,0.400,3,0.600,0.000,1.000,0.500,0.500
25,Rhodothermia,1,5,2,0.400,3,0.600,0.000,1.000,0.500,0.500
22,Myxococcia,1,10,4,0.400,6,0.600,0.000,1.000,0.444,0.556
16,Halobacteria,2,18,7,0.389,11,0.611,0.200,0.800,0.462,0.538
11,Deinococci,1,11,4,0.364,7,0.636,0.333,0.667,0.375,0.625
10,Cyanophyceae,3,36,13,0.361,23,0.639,0.385,0.615,0.300,0.700
15,Gammaproteobacteria,26,356,123,0.346,233,0.654,0.422,0.578,0.240,0.760
4,Betaproteobacteria,2,19,6,0.316,13,0.684,0.111,0.889,0.500,0.500
12,Epsilonproteobacteria,1,10,3,0.300,7,0.700,0.500,0.500,0.250,0.750



=== By order ===


,order,n_species,total_pairs,total_errors,error_rate,total_correct,correct_rate,fn_rate,tp_rate,fp_rate,tn_rate
31,Micrococcales,1,3,2,0.667,1,0.333,0.000,1.000,1.000,0.000
11,Chlamydiales,2,9,5,0.556,4,0.444,0.000,1.000,0.714,0.286
42,Pseudomonadales,2,112,57,0.509,55,0.491,0.515,0.485,0.455,0.545
47,Synechococcales,1,23,10,0.435,13,0.565,0.429,0.571,0.500,0.500
35,Myxococcales,1,10,4,0.400,6,0.600,0.000,1.000,0.444,0.556
45,Rhodothermales,1,5,2,0.400,3,0.600,0.000,1.000,0.500,0.500
30,Methanobacteriales,1,5,2,0.400,3,0.600,0.000,1.000,0.500,0.500
24,Isochrysidales,1,5,2,0.400,3,0.600,0.000,1.000,0.500,0.500
23,Hyphomicrobiales,6,43,17,0.395,26,0.605,0.125,0.875,0.457,0.543
22,Halobacteriales,2,18,7,0.389,11,0.611,0.200,0.800,0.462,0.538



=== By family ===


,family,n_species,total_pairs,total_errors,error_rate,total_correct,correct_rate,fn_rate,tp_rate,fp_rate,tn_rate
35,Micrococcaceae,1,3,2,0.667,1,0.333,0.000,1.000,1.000,0.000
14,Chlamydiaceae,2,9,5,0.556,4,0.444,0.000,1.000,0.714,0.286
54,Pseudomonadaceae,2,112,57,0.509,55,0.491,0.515,0.485,0.455,0.545
50,Phyllobacteriaceae,1,4,2,0.500,2,0.500,1.000,0.000,0.333,0.667
42,Natrialbaceae,1,11,5,0.455,6,0.545,1.000,0.000,0.400,0.600
51,Prochlorococcaceae,1,23,10,0.435,13,0.565,0.429,0.571,0.500,0.500
18,Comamonadaceae,1,7,3,0.429,4,0.571,0.000,1.000,0.600,0.400
19,Corynebacteriaceae,3,21,9,0.429,12,0.571,0.182,0.818,0.700,0.300
44,Noelaerhabdaceae,1,5,2,0.400,3,0.600,0.000,1.000,0.500,0.500
34,Methanobacteriaceae,1,5,2,0.400,3,0.600,0.000,1.000,0.500,0.500


In [15]:
# --- Per-phage counts split by correctness and interaction type ---
# Reuses get_taxids / get_taxonomy defined above

def phage_counts_by_type(df, label):
    return df[df["interaction_type"] == label]["phage_id"].value_counts()

errors_ph  = dataset[dataset["interaction_type"] != dataset["prediction"]]
correct_ph = dataset[dataset["interaction_type"] == dataset["prediction"]]

ptotal_pos   = phage_counts_by_type(dataset,    1)
ptotal_neg   = phage_counts_by_type(dataset,    0)
perrors_pos  = phage_counts_by_type(errors_ph,  1)   # FN
perrors_neg  = phage_counts_by_type(errors_ph,  0)   # FP
pcorrect_pos = phage_counts_by_type(correct_ph, 1)   # TP
pcorrect_neg = phage_counts_by_type(correct_ph, 0)   # TN

ptotal_counts = dataset["phage_id"].value_counts()
perror_counts = errors_ph["phage_id"].value_counts()
phage_accessions = perror_counts.index.tolist()

print(f"Fetching taxonomy for {len(phage_accessions)} unique phage accessions...")
phage_taxid_map = get_taxids(phage_accessions)
missing_ph = [acc for acc in phage_accessions if acc not in phage_taxid_map]
if missing_ph:
    print(f"Warning: no TaxID found for {len(missing_ph)} accessions: {missing_ph}")
phage_tax_map = get_taxonomy(phage_taxid_map)
print("Done.")

phage_rows = []
for acc in phage_accessions:
    taxid = phage_taxid_map.get(acc)
    tax   = phage_tax_map.get(taxid, {}) if taxid else {}

    tot      = ptotal_counts.get(acc, 0)
    tot_pos  = ptotal_pos.get(acc, 0)
    tot_neg  = ptotal_neg.get(acc, 0)
    err      = perror_counts.get(acc, 0)
    err_pos  = perrors_pos.get(acc, 0)
    err_neg  = perrors_neg.get(acc, 0)
    cor_pos  = pcorrect_pos.get(acc, 0)
    cor_neg  = pcorrect_neg.get(acc, 0)
    cor      = cor_pos + cor_neg

    phage_rows.append({
        "phage_id":     acc,
        # overall
        "errors":       err,
        "correct":      cor,
        "total":        tot,
        "error_rate":   round(err / tot, 3) if tot else None,
        "correct_rate": round(cor / tot, 3) if tot else None,
        # positive (FN/TP)
        "errors_pos":   err_pos,
        "correct_pos":  cor_pos,
        "total_pos":    tot_pos,
        "fn_rate":      round(err_pos / tot_pos, 3) if tot_pos else None,
        "tp_rate":      round(cor_pos / tot_pos, 3) if tot_pos else None,
        # negative (FP/TN)
        "errors_neg":   err_neg,
        "correct_neg":  cor_neg,
        "total_neg":    tot_neg,
        "fp_rate":      round(err_neg / tot_neg, 3) if tot_neg else None,
        "tn_rate":      round(cor_neg / tot_neg, 3) if tot_neg else None,
        # taxonomy (viral ranks)
        "species":      tax.get("species",      "N/A"),
        "genus":        tax.get("genus",        "N/A"),
        "family":       tax.get("family",       "N/A"),
        "order":        tax.get("order",        "N/A"),
        "class":        tax.get("class",        "N/A"),
        "phylum":       tax.get("phylum",       "N/A"),
        "superkingdom": tax.get("superkingdom", "N/A"),
    })

phage_taxonomy_df = pd.DataFrame(phage_rows)
phage_taxonomy_df


Fetching taxonomy for 254 unique phage accessions...


Done.


,phage_id,errors,correct,total,error_rate,correct_rate,errors_pos,correct_pos,total_pos,fn_rate,...,total_neg,fp_rate,tn_rate,species,genus,family,order,class,phylum,superkingdom
0,NC_015290,4,1,5,0.800,0.200,1,0,1,1.0,...,4,0.75,0.25,Prochlorococcus phage P-SSM7,Palaemonvirus,Kyanoviridae,Pantevenvirales,Caudoviricetes,Uroviricota,N/A
1,NC_017972,3,0,3,1.000,0.000,1,0,1,1.0,...,2,1.00,0.00,Pseudomonas phage Lu11,N/A,N/A,N/A,Caudoviricetes,Uroviricota,N/A
2,NC_028799,3,3,6,0.500,0.500,1,0,1,1.0,...,5,0.40,0.60,Vibrio phage phi 1,Pacinivirus,Schitoviridae,N/A,Caudoviricetes,Uroviricota,N/A
3,NC_022974,3,2,5,0.600,0.400,1,0,1,1.0,...,4,0.50,0.50,Pseudomonas phage CHA_P1,Nankokuvirus,Vandenendeviridae,N/A,Caudoviricetes,Uroviricota,N/A
4,NC_016563,3,2,5,0.600,0.400,0,1,1,0.0,...,4,0.75,0.25,Bacillus phage W.Ph.,Wphvirus,Herelleviridae,N/A,Caudoviricetes,Uroviricota,N/A
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
249,NC_028999,1,0,1,1.000,0.000,1,0,1,1.0,...,0,NaN,NaN,Pseudomonas phage PhiPA3,Miltoncavirus,Chimalliviridae,N/A,Caudoviricetes,Uroviricota,N/A
250,NC_019515,1,0,1,1.000,0.000,1,0,1,1.0,...,0,NaN,NaN,Bacillus phage BCD7,Becedseptimavirus,N/A,N/A,Caudoviricetes,Uroviricota,N/A
251,NC_023841,1,3,4,0.250,0.750,1,0,1,1.0,...,3,0.00,1.00,Dunaliella viridis virus SI2,N/A,N/A,N/A,Caudoviricetes,Uroviricota,N/A
252,NC_005887,1,2,3,0.333,0.667,0,1,1,0.0,...,2,0.50,0.50,Burkholderia phage BcepC6B,Ryyoungvirus,N/A,N/A,Caudoviricetes,Uroviricota,N/A


In [16]:
phage_taxonomy_df.to_csv('./error_analysis_by_phage.csv', index=False)

In [17]:
# --- Rank summaries for phages ---
# rank_summary is generic — reuse it directly on phage_taxonomy_df

# Viral taxonomy uses family/order/class as most informative levels
for rank in ["family", "order", "class"]:
    summary = rank_summary(phage_taxonomy_df, rank)
    cols = [rank, "n_species", "total_pairs", "total_errors", "error_rate",
            "total_correct", "correct_rate", "fn_rate", "tp_rate", "fp_rate", "tn_rate"]
    available = [c for c in cols if c in summary.columns]
    print(f"\n=== By {rank} ===")
    display(summary[available].head(15))



=== By family ===


,family,n_species,total_pairs,total_errors,error_rate,total_correct,correct_rate,fn_rate,tp_rate,fp_rate,tn_rate
23,N/A,125,303,150,0.495,153,0.505,0.496,0.504,0.494,0.506
16,Herelleviridae,20,46,25,0.543,21,0.457,0.700,0.300,0.423,0.577
2,Autoscriptoviridae,19,42,25,0.595,17,0.405,0.789,0.211,0.435,0.565
31,Schitoviridae,8,25,12,0.480,13,0.520,0.500,0.500,0.471,0.529
37,Vandenendeviridae,8,21,10,0.476,11,0.524,0.875,0.125,0.231,0.769
17,Inoviridae,7,20,9,0.450,11,0.550,0.571,0.429,0.385,0.615
18,Kyanoviridae,4,13,8,0.615,5,0.385,1.000,0.000,0.444,0.556
19,Lindbergviridae,7,19,7,0.368,12,0.632,0.286,0.714,0.417,0.583
35,Straboviridae,6,13,6,0.462,7,0.538,0.500,0.500,0.429,0.571
40,Zobellviridae,3,11,5,0.455,6,0.545,1.000,0.000,0.250,0.750



=== By order ===


,order,n_species,total_pairs,total_errors,error_rate,total_correct,correct_rate,fn_rate,tp_rate,fp_rate,tn_rate
7,N/A,196,474,233,0.492,241,0.508,0.531,0.469,0.464,0.536
1,Autographivirales,28,64,36,0.562,28,0.438,0.750,0.250,0.417,0.583
9,Pantevenvirales,10,26,14,0.538,12,0.462,0.700,0.300,0.438,0.562
10,Tubulavirales,7,20,9,0.450,11,0.550,0.571,0.429,0.385,0.615
3,Herpesvirales,2,6,4,0.667,2,0.333,0.000,1.000,1.000,0.000
2,Halopanivirales,3,9,3,0.333,6,0.667,0.000,1.000,0.500,0.500
8,Norzivirales,2,6,2,0.333,4,0.667,0.500,0.500,0.250,0.750
4,Imitervirales,2,5,2,0.400,3,0.600,0.000,1.000,0.667,0.333
6,Lefavirales,1,2,2,1.000,0,0.000,1.000,0.000,1.000,0.000
0,Algavirales,1,2,1,0.500,1,0.500,0.000,1.000,1.000,0.000



=== By class ===


,class,n_species,total_pairs,total_errors,error_rate,total_correct,correct_rate,fn_rate,tp_rate,fp_rate,tn_rate
1,Caudoviricetes,232,561,281,0.501,280,0.499,0.560,0.440,0.459,0.541
2,Faserviricetes,7,20,9,0.450,11,0.550,0.571,0.429,0.385,0.615
3,Herviviricetes,2,6,4,0.667,2,0.333,0.000,1.000,1.000,0.000
4,Laserviricetes,3,9,3,0.333,6,0.667,0.000,1.000,0.500,0.500
6,Megaviricetes,3,7,3,0.429,4,0.571,0.000,1.000,0.750,0.250
5,Leviviricetes,2,6,2,0.333,4,0.667,0.500,0.500,0.250,0.750
8,Naldaviricetes,1,2,2,1.000,0,0.000,1.000,0.000,1.000,0.000
7,N/A,2,3,2,0.667,1,0.333,1.000,0.000,0.000,1.000
0,Belvinaviricetes,1,4,1,0.250,3,0.750,0.000,1.000,0.333,0.667
9,Tectiliviricetes,1,2,1,0.500,1,0.500,1.000,0.000,0.000,1.000


In [18]:
# bact by family with total > 20 and ordered by error rate
bact_family_summary = rank_summary(taxonomy_df, "family")
bact_family_summary = bact_family_summary[bact_family_summary["total_pairs"] > 20]
bact_family_summary = bact_family_summary.sort_values("error_rate", ascending=False)
bact_family_summary[["family", "n_species", "total_pairs", "error_rate", "fp_rate", "fn_rate"]].head(40)

,family,n_species,total_pairs,error_rate,fp_rate,fn_rate
54,Pseudomonadaceae,2,112,0.509,0.455,0.515
51,Prochlorococcaceae,1,23,0.435,0.500,0.429
19,Corynebacteriaceae,3,21,0.429,0.700,0.182
37,Moraxellaceae,3,44,0.386,0.412,0.370
55,Rhizobiaceae,5,39,0.385,0.469,0.000
39,Mycobacteriaceae,3,35,0.371,0.391,0.333
60,Streptococcaceae,3,25,0.360,0.450,0.000
66,Vibrionaceae,3,50,0.340,0.158,0.452
31,Lysobacteraceae,3,24,0.292,0.125,0.625
23,Enterobacteriaceae,5,42,0.262,0.250,0.278


In [20]:
"""
Verification script for Taxonomic Error Analysis subsection.
"""

import pandas as pd
import sys

# Load data
bact = pd.read_csv("error_analysis_by_bacterium.csv")
phage = pd.read_csv("error_analysis_by_phage.csv")

# Ground truth from confusion matrix (full test set)
CM_TP, CM_FP, CM_FN, CM_TN = 479, 169, 139, 449
CM_TOTAL = CM_TP + CM_FP + CM_FN + CM_TN  # 1236

print("=" * 80)
print("TAXONOMIC ERROR ANALYSIS SECTION")
print("=" * 80)

# ============================================================
# CSV now covers all 1,236 test pairs
# ============================================================
total_pairs = bact["total"].sum()
total_errors = bact["errors"].sum()
overall_error_rate = total_errors / total_pairs

print(f"\n--- Overall stats ---")
print(f"  Confusion matrix total: {CM_TOTAL} pairs")
print(f"  CSV total:              {total_pairs} pairs")
print(f"  Errors: {total_errors}, error rate: {overall_error_rate:.1%}")

assert total_pairs == CM_TOTAL, f"MISMATCH: CSV has {total_pairs} pairs, confusion matrix has {CM_TOTAL}"
assert total_errors == CM_FN + CM_FP, f"MISMATCH: CSV errors={total_errors}, CM FP+FN={CM_FN + CM_FP}"
print(f"  ✓ CSV total matches confusion matrix ({CM_TOTAL} pairs)")

# ============================================================
# Family-level aggregation (n>=20)
# Streptomycetaceae vs Pseudomonadaceae
# ============================================================
print(f"\n--- Family-level error rates ---")

bfam = bact.groupby("family").agg(
    n_species=("species", "nunique"),
    total_pairs=("total", "sum"),
    total_errors=("errors", "sum"),
    total_correct=("correct", "sum"),
    total_fn=("errors_pos", "sum"),
    total_tp=("correct_pos", "sum"),
    total_fp=("errors_neg", "sum"),
    total_tn=("correct_neg", "sum")
).reset_index()
bfam["error_rate"] = bfam["total_errors"] / bfam["total_pairs"]
bfam["fn_rate"] = bfam["total_fn"] / (bfam["total_fn"] + bfam["total_tp"])
bfam["fp_rate"] = bfam["total_fp"] / (bfam["total_fp"] + bfam["total_tn"])
bfam_filtered = bfam[bfam["total_pairs"] >= 20].sort_values("error_rate", ascending=False)

strep  = bfam_filtered[bfam_filtered["family"] == "Streptomycetaceae"].iloc[0]
pseudo = bfam_filtered[bfam_filtered["family"] == "Pseudomonadaceae"].iloc[0]

print(f"  Streptomycetaceae: n={strep['total_pairs']}, errors={strep['total_errors']}, rate={strep['error_rate']:.3f}")
print(f"  Pseudomonadaceae:  n={pseudo['total_pairs']}, errors={pseudo['total_errors']}, rate={pseudo['error_rate']:.3f}")
ratio = pseudo["error_rate"] / strep["error_rate"]
print(f"  Ratio: {ratio:.1f}x")

# ============================================================
# P. aeruginosa: 53 of 308 errors (17.2%),
#           error rate 52.5%, fn rate 50.5%
# ============================================================
print(f"\n--- P. aeruginosa stats ---")

pa = bact[bact["bacterium_id"] == "HG530068"].iloc[0]
pa_pct = pa["errors"] / total_errors

print(f"  Errors: {pa['errors']}, total: {pa['total']}, error_rate: {pa['errors']/pa['total']:.3f}")
print(f"  Fraction of all errors: {pa_pct:.3f}")
fn_rate_pa = pa["errors_pos"] / (pa["errors_pos"] + pa["correct_pos"])
print(f"  FN rate: {fn_rate_pa:.3f}")
print(f"  Text says: 53 of 308 (17.2%), error rate 52.5%, fn rate 50.5%")

assert pa["errors"] == 53, f"MISMATCH: errors={pa['errors']}"
assert round(pa_pct, 3) == 0.172, f"MISMATCH: pct={pa_pct:.3f}"
assert round(pa["errors"] / pa["total"], 3) == 0.525, f"MISMATCH: error_rate={pa['errors']/pa['total']:.3f}"
assert round(fn_rate_pa, 3) == 0.505, f"MISMATCH: fn_rate={fn_rate_pa:.3f}"
print(f"  ✓ VERIFIED")

# ============================================================
# Top 3 species — update pairs_pct denominator to 1236
# ============================================================
print(f"\n--- Top 3 species ---")

top3 = bact.nlargest(3, "errors")
top3_errors = top3["errors"].sum()
top3_pairs = top3["total"].sum()
top3_error_pct = top3_errors / total_errors
top3_pairs_pct = top3_pairs / total_pairs

print(f"  Top 3 species: {list(top3['species'].values)}")
print(f"  Top 3 errors: {top3_errors} ({top3_error_pct:.1%} of all errors)")
print(f"  Top 3 pairs:  {top3_pairs} ({top3_pairs_pct:.1%} of all pairs)")

assert top3_errors == 77, f"MISMATCH: errors={top3_errors}"
assert round(top3_error_pct, 3) == 0.250, f"MISMATCH: error_pct={top3_error_pct:.3f}"
top3_classes = top3["class"].unique()
assert all(top3["class"] == "Gammaproteobacteria"), f"MISMATCH: {top3_classes}"
print(f"  ✓ VERIFIED (77 errors = 25.0%, all Gammaproteobacteria)")
print(f"  NOTE: pairs_pct is now {top3_pairs_pct:.1%} (was 13.7% over 1,135 — update text to {top3_pairs_pct:.1%} over 1,236)")

# ============================================================
# Pseudomonadaceae fn_rate 51.5%, Vibrionaceae 45.2%
# ============================================================
print(f"\n--- FN-dominated families ---")

vibrio = bfam_filtered[bfam_filtered["family"] == "Vibrionaceae"].iloc[0]

print(f"  Pseudomonadaceae fn_rate: {pseudo['fn_rate']:.3f}")
print(f"  Vibrionaceae fn_rate: {vibrio['fn_rate']:.3f}")

assert round(pseudo["fn_rate"], 3) == 0.515, f"MISMATCH: {pseudo['fn_rate']:.3f}"
assert round(vibrio["fn_rate"], 3) == 0.452, f"MISMATCH: {vibrio['fn_rate']:.3f}"
print(f"  ✓ VERIFIED")

# ============================================================
# Corynebacteriaceae fp_rate 70.0%, Rhizobiaceae 46.9%
# ============================================================
print(f"\n--- FP-dominated families ---")

coryne = bfam_filtered[bfam_filtered["family"] == "Corynebacteriaceae"].iloc[0]
rhizo  = bfam_filtered[bfam_filtered["family"] == "Rhizobiaceae"].iloc[0]

print(f"  Corynebacteriaceae fp_rate: {coryne['fp_rate']:.3f}")
print(f"  Rhizobiaceae fp_rate: {rhizo['fp_rate']:.3f}")

assert round(coryne["fp_rate"], 3) == 0.700, f"MISMATCH: {coryne['fp_rate']:.3f}"
assert round(rhizo["fp_rate"], 3) == 0.469, f"MISMATCH: {rhizo['fp_rate']:.3f}"
print(f"  ✓ VERIFIED")

# ============================================================
# Phage side - Kyanoviridae 61.5%, Autoscriptoviridae 59.5%
# ============================================================
print(f"\n--- Phage family error rates ---")

pfam = phage.groupby("family").agg(
    n_species=("species", "nunique"),
    total_pairs=("total", "sum"),
    total_errors=("errors", "sum"),
).reset_index()
pfam["error_rate"] = pfam["total_errors"] / pfam["total_pairs"]
pfam_filtered = pfam[pfam["total_pairs"] >= 10].sort_values("error_rate", ascending=False)

kyano = pfam_filtered[pfam_filtered["family"] == "Kyanoviridae"].iloc[0]
auto  = pfam_filtered[pfam_filtered["family"] == "Autoscriptoviridae"].iloc[0]

print(f"  Kyanoviridae: n={kyano['total_pairs']}, rate={kyano['error_rate']:.3f}")
print(f"  Autoscriptoviridae: n={auto['total_pairs']}, rate={auto['error_rate']:.3f}")

assert round(kyano["error_rate"], 3) == 0.615, f"MISMATCH: {kyano['error_rate']:.3f}"
assert round(auto["error_rate"], 3) == 0.595, f"MISMATCH: {auto['error_rate']:.3f}"
print(f"  ✓ VERIFIED")

# ============================================================
# FULL TABLE DUMP
# ============================================================
print(f"\n--- FULL TABLE (families with n>=20, sorted by error rate) ---")
print(bfam_filtered[["family", "n_species", "total_pairs", "total_errors", "error_rate", "fn_rate", "fp_rate"]].to_string(index=False))

print("\n" + "=" * 80)
print("ALL CLAIMS VERIFIED SUCCESSFULLY")
print(f"Total pairs in CSV: {total_pairs} (matches confusion matrix: {total_pairs == CM_TOTAL})")
print(f"Overall error rate: {overall_error_rate:.1%} ({total_errors}/{total_pairs})")
print("=" * 80)


TAXONOMIC ERROR ANALYSIS SECTION

--- Overall stats ---
  Confusion matrix total: 1236 pairs
  CSV total:              1236 pairs
  Errors: 308, error rate: 24.9%
  ✓ CSV total matches confusion matrix (1236 pairs)

--- Family-level error rates ---
  Streptomycetaceae: n=142, errors=15, rate=0.106
  Pseudomonadaceae:  n=112, errors=57, rate=0.509
  Ratio: 4.8x

--- P. aeruginosa stats ---
  Errors: 53, total: 101, error_rate: 0.525
  Fraction of all errors: 0.172
  FN rate: 0.505
  Text says: 53 of 308 (17.2%), error rate 52.5%, fn rate 50.5%
  ✓ VERIFIED

--- Top 3 species ---
  Top 3 species: ['Pseudomonas aeruginosa PA38182', 'Vibrio cholerae', 'Acinetobacter baumannii']
  Top 3 errors: 77 (25.0% of all errors)
  Top 3 pairs:  156 (12.6% of all pairs)
  ✓ VERIFIED (77 errors = 25.0%, all Gammaproteobacteria)
  NOTE: pairs_pct is now 12.6% (was 13.7% over 1,135 — update text to 12.6% over 1,236)

--- FN-dominated families ---
  Pseudomonadaceae fn_rate: 0.515
  Vibrionaceae fn_rate: 